# Attention Variants from Scratch

This notebook implements the efficiency variants that power today's frontier LLMs — building on the four foundations (Self-Attention, Multi-Head Attention, Causal Self-Attention, Cross-Attention) from the earlier notebooks.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Every variant below modifies *how* Q, K, V are produced, *which* tokens attend to which, or *what* happens to the attention output — but the core equation stays the same.

| Variant | What it changes | Bottleneck addressed |
|---------|----------------|---------------------|
| **GQA** | Shares K/V across groups of query heads | KV cache memory |
| **MLA** | Compresses K/V into low-rank latent before caching | KV cache memory (aggressive) |
| **SWA** | Restricts each token to a local window | Sequence length scaling |
| **NSA** | Dynamically selects which tokens to attend to | Sequence length scaling (learned) |
| **Gated Attention** | Adds a learned gate on the attention output | Training stability |
| **Hybrid** | Interleaves linear and full attention layers | Total compute budget |

## 1. Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## 2. Grouped-Query Attention (GQA)

Standard MHA uses $h$ independent K/V heads — one per query head. GQA reduces this to $h_{kv}$ shared K/V heads, where each K/V head serves a **group** of $h / h_{kv}$ query heads. Cache shrinks by the group ratio with minimal quality loss.

$$\text{GQA:}\quad Q_i \in \mathbb{R}^{d_k},\quad K_g, V_g \in \mathbb{R}^{d_k} \quad\text{where}\; g = \lfloor i \cdot h_{kv} / h \rfloor$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| $Q$ | $(\text{batch},\, h,\, \text{seq\_len},\, d_k)$ | All query heads — independent projections |
| $K, V$ | $(\text{batch},\, h_{kv},\, \text{seq\_len},\, d_k)$ | Shared K/V heads |
| `group_size` | $h / h_{kv}$ | Query heads per K/V group |
| KV cache | $2 \times h_{kv} \times T \times d_k$ | Reduced by factor $h / h_{kv}$ vs MHA |

> **Spectrum:** MHA ($h_{kv} = h$) → GQA ($1 < h_{kv} < h$) → MQA ($h_{kv} = 1$). LLaMA 3 uses $h=32, h_{kv}=8$ — a 4× cache reduction.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (GQA).

    Groups n_heads query heads to share n_kv_heads K/V heads, reducing KV
    cache by a factor of n_heads / n_kv_heads with minimal quality loss.

    GQA(X) = Concat(head_1, ..., head_h) W_O
    where head_i uses K_g, V_g with g = i * n_kv_heads // n_heads
    """

    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        """
        Args:
            d_model:    Input embedding dimension. Must be divisible by n_heads.
            n_heads:    Number of query heads.
            n_kv_heads: Number of K/V heads. Must divide n_heads evenly.
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        # n_heads independent query projections, n_kv_heads shared K/V projections
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x:    Input tensor, shape (batch, seq_len, d_model).
            mask: Optional boolean mask, shape (seq_len, seq_len).

        Returns:
            out:     Context vectors, shape (batch, seq_len, d_model).
            weights: Attention weights, shape (batch, n_heads, seq_len, seq_len).
        """
        B, T, _ = x.shape

        # Project and reshape into heads
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)       # (B, n_heads, T, d_k)
        K = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)    # (B, n_kv_heads, T, d_k)
        V = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)    # (B, n_kv_heads, T, d_k)

        # Expand K/V to match n_heads by repeating each KV head group_size times
        K = K.repeat_interleave(self.group_size, dim=1)  # (B, n_heads, T, d_k)
        V = V.repeat_interleave(self.group_size, dim=1)  # (B, n_heads, T, d_k)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, n_heads, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)               # (B, n_heads, T, T)
        context = torch.matmul(weights, V)                 # (B, n_heads, T, d_k)

        # Concatenate heads and project
        context = context.transpose(1, 2).contiguous().view(B, T, -1)  # (B, T, d_model)
        out = self.W_o(context)                                         # (B, T, d_model)
        return out, weights

In [ ]:
torch.manual_seed(0)

batch, seq_len, d_model = 2, 7, 64
n_heads, n_kv_heads = 8, 2  # group_size = 4

x = torch.randn(batch, seq_len, d_model)
gqa = GroupedQueryAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads)
out, weights = gqa(x)

print(f"Input   shape: {x.shape}")        # (2, 7, 64)
print(f"Output  shape: {out.shape}")      # (2, 7, 64)
print(f"Weights shape: {weights.shape}")  # (2, 8, 7, 7)
print(f"d_k per head : {gqa.d_k}")       # 8
print(f"Group size   : {gqa.group_size}") # 4

# Causal mask
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
out_causal, w_causal = gqa(x, mask=causal_mask)
assert (w_causal[:, :, causal_mask] < 1e-6).all(), "Causal mask not applied correctly!"

# KV cache comparison: GQA vs MHA
kv_cache_mha = 2 * n_heads * seq_len * gqa.d_k      # full MHA
kv_cache_gqa = 2 * n_kv_heads * seq_len * gqa.d_k   # GQA
print(f"\nKV cache per layer (elements): MHA={kv_cache_mha}, GQA={kv_cache_gqa}")
print(f"Cache reduction: {kv_cache_mha / kv_cache_gqa:.0f}×")
print("\nShape and causal mask checks passed.")

## 3. Multi-Head Latent Attention (MLA)

Instead of caching full K/V per head, MLA compresses them into a low-rank **latent** $c$ before caching, then decompresses during attention. Only $c$ is stored — not the full K and V.

$$c = X W_{\text{down}}, \quad K = c\, W_{\text{up}}^K, \quad V = c\, W_{\text{up}}^V$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| $W_{\text{down}}$ | $(d_{\text{model}},\, d_{\text{latent}})$ | Down-projection into bottleneck |
| $c$ | $(\text{batch},\, \text{seq\_len},\, d_{\text{latent}})$ | **Cached** — the only thing stored per position |
| $W_{\text{up}}^K, W_{\text{up}}^V$ | $(d_{\text{latent}},\, h \cdot d_k)$ | Up-project back to full K/V at attention time |

> **Absorption trick:** at inference, $Q K^\top = (X W_Q)(c\, W_{\text{up}}^K)^\top = X\,(W_Q {W_{\text{up}}^K}^\top)\, c^\top$. Pre-compute $W_{\text{absorbed}} = W_Q {W_{\text{up}}^K}^\top$ once — K is never materialized.

In [ ]:
class MultiHeadLatentAttention(nn.Module):
    """
    Multi-Head Latent Attention (MLA).

    Down-projects K/V into a shared low-rank latent c (d_latent << d_model)
    before caching, then up-projects back to full K/V during attention.
    Only c is cached — dramatically smaller than per-head K/V tensors.

    c = X @ W_down                  (compress)
    K = c @ W_up_K, V = c @ W_up_V (decompress)
    Attention proceeds as standard multi-head.
    """

    def __init__(self, d_model: int, n_heads: int, d_latent: int):
        """
        Args:
            d_model:  Input embedding dimension. Must be divisible by n_heads.
            n_heads:  Number of attention heads.
            d_latent: Latent bottleneck dimension (compression target).
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_latent = d_latent

        # Query: standard projection
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)

        # KV: down-project to latent, up-project to full K/V
        self.W_down = nn.Linear(d_model, d_latent, bias=False)
        self.W_up_k = nn.Linear(d_latent, n_heads * self.d_k, bias=False)
        self.W_up_v = nn.Linear(d_latent, n_heads * self.d_k, bias=False)

        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x:    Input tensor, shape (batch, seq_len, d_model).
            mask: Optional boolean mask, shape (seq_len, seq_len).

        Returns:
            out:     Context vectors, shape (batch, seq_len, d_model).
            weights: Attention weights, shape (batch, n_heads, seq_len, seq_len).
        """
        B, T, _ = x.shape

        # Query — standard multi-head projection
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)

        # KV — compress to latent, then decompress
        c = self.W_down(x)          # (B, T, d_latent) — THIS is what gets cached
        K = self.W_up_k(c).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)
        V = self.W_up_v(c).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, n_heads, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)               # (B, n_heads, T, T)
        context = torch.matmul(weights, V)                 # (B, n_heads, T, d_k)

        # Concatenate heads and project
        context = context.transpose(1, 2).contiguous().view(B, T, -1)  # (B, T, d_model)
        out = self.W_o(context)                                         # (B, T, d_model)
        return out, weights

In [ ]:
torch.manual_seed(0)

batch, seq_len, d_model = 2, 7, 64
n_heads, d_latent = 4, 16  # compress from 64 to 16

x = torch.randn(batch, seq_len, d_model)
mla = MultiHeadLatentAttention(d_model=d_model, n_heads=n_heads, d_latent=d_latent)
out, weights = mla(x)

print(f"Input   shape: {x.shape}")        # (2, 7, 64)
print(f"Output  shape: {out.shape}")      # (2, 7, 64)
print(f"Weights shape: {weights.shape}")  # (2, 4, 7, 7)
print(f"d_k per head : {mla.d_k}")       # 16
print(f"d_latent     : {mla.d_latent}")   # 16

# Cache comparison: MLA vs MHA vs GQA (per layer, per position)
cache_mha = 2 * n_heads * mla.d_k         # full K + V per position
cache_gqa = 2 * 2 * mla.d_k               # GQA with n_kv_heads=2
cache_mla = d_latent                        # just the latent c
print(f"\nCache per position (elements): MHA={cache_mha}, GQA(n_kv=2)={cache_gqa}, MLA={cache_mla}")
print(f"MLA compression ratio: {cache_mha / cache_mla:.1f}×")

# Verify causal mask works
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
_, w_causal = mla(x, mask=causal_mask)
assert (w_causal[:, :, causal_mask] < 1e-6).all()
print("\nShape, cache, and causal checks passed.")

## 4. Sliding Window Attention (SWA)

Each token attends only to its $w$ nearest predecessors instead of the full sequence. The attention matrix becomes **band-diagonal** — $O(n \cdot w)$ instead of $O(n^2)$.

$$\text{SWA: token } i \text{ attends to } \max(0,\, i - w + 1) \ldots i$$

| Property | Full Causal | SWA |
|----------|------------|-----|
| Entries per row | $i + 1$ | $\min(i + 1,\, w)$ |
| Total active entries | $T(T+1)/2$ | $\approx T \cdot w$ |
| KV cache per layer | $T \cdot d_k$ (grows) | $w \cdot d_k$ (fixed) |
| Effective receptive field | $T$ (single layer) | $L \cdot w$ (after $L$ layers) |

> **Rolling buffer cache:** store keys/values at `position mod w` — bounded memory regardless of sequence length.

In [ ]:
class SlidingWindowAttention(nn.Module):
    """
    Sliding Window Attention (SWA).

    Restricts each token to attending only to its w most recent predecessors
    (including itself). Produces a band-diagonal attention matrix instead of
    the full lower-triangle, reducing complexity from O(n^2) to O(n * w).

    Mask: causal (upper-triangle blocked) AND window (positions beyond w blocked).
    """

    def __init__(self, d_model: int, n_heads: int, window_size: int):
        """
        Args:
            d_model:     Input embedding dimension. Must be divisible by n_heads.
            n_heads:     Number of attention heads.
            window_size: Number of recent tokens each position can attend to.
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.window_size = window_size

        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)

    def _build_swa_mask(self, seq_len: int) -> torch.Tensor:
        """
        Build a boolean mask that is True for positions to BLOCK.
        Combines causal mask (block future) with window mask (block beyond w).

        Returns:
            mask: Boolean tensor, shape (seq_len, seq_len). True = blocked.
        """
        # Start with causal: block future positions (upper triangle)
        causal = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
        # Block positions beyond window (lower triangle past w)
        window = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=-(self.window_size))
        return causal | window  # True = blocked

    def forward(
        self,
        x: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: Input tensor, shape (batch, seq_len, d_model).

        Returns:
            out:     Context vectors, shape (batch, seq_len, d_model).
            weights: Attention weights, shape (batch, n_heads, seq_len, seq_len).
        """
        B, T, _ = x.shape

        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, n_heads, T, T)

        mask = self._build_swa_mask(T).to(x.device)
        scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)               # (B, n_heads, T, T)
        context = torch.matmul(weights, V)                 # (B, n_heads, T, d_k)

        context = context.transpose(1, 2).contiguous().view(B, T, -1)  # (B, T, d_model)
        out = self.W_o(context)                                         # (B, T, d_model)
        return out, weights

In [ ]:
torch.manual_seed(0)

batch, seq_len, d_model = 2, 16, 64
n_heads, w = 4, 4

x = torch.randn(batch, seq_len, d_model)
swa = SlidingWindowAttention(d_model=d_model, n_heads=n_heads, window_size=w)
out, weights = swa(x)

print(f"Input   shape: {x.shape}")        # (2, 16, 64)
print(f"Output  shape: {out.shape}")      # (2, 16, 64)
print(f"Weights shape: {weights.shape}")  # (2, 4, 16, 16)
print(f"Window size  : {w}")

# Print mask to verify band structure
mask = swa._build_swa_mask(seq_len)
active = (~mask).sum().item()
total = seq_len * seq_len
print(f"\nActive entries: {active}/{total} ({100 * active / total:.1f}%)")
print(f"Full causal would be: {seq_len * (seq_len + 1) // 2}/{total}")
print(f"\nMask (0=attend, 1=blocked):")
print(mask.int())

In [ ]:
# Visualize: full causal mask vs SWA mask (w=4) for T=16
T = 16
causal = ~torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # True = attend
swa_active = ~swa._build_swa_mask(T)                                   # True = attend

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, matrix, title in [
    (axes[0], causal.float().numpy(), "Full Causal Attention"),
    (axes[1], swa_active.float().numpy(), f"Sliding Window (w={w})"),
]:
    im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xlabel("Key position", fontsize=11)
    ax.set_ylabel("Query position", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{matrix[i, j]:.0f}", ha="center", va="center",
                    fontsize=6, color="white" if matrix[i, j] > 0.5 else "black")

plt.suptitle("Attention Masks: Full Causal vs. Sliding Window", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Native Sparse Attention (NSA)

NSA combines three parallel attention paths — each capturing a different granularity of context — and gates their outputs per-head:

$$\text{out} = g_{\text{cmp}} \cdot A_{\text{compressed}} + g_{\text{sel}} \cdot A_{\text{selected}} + g_{\text{swa}} \cdot A_{\text{sliding}}$$

| Path | What it does | Cost |
|------|-------------|------|
| **Compressed** | Block-mean pool K/V by factor $b$, attend to $T/b$ summaries | $O(n \cdot n/b)$ |
| **Selected** | Score blocks, select top-$k$ most relevant, run full attention on those | $O(n \cdot k \cdot b)$ |
| **Sliding window** | Standard local attention within window $w$ | $O(n \cdot w)$ |

> **Key insight:** sparse attention trained *natively* (not applied post-hoc) matches or exceeds full dense attention quality — with 6–11× speedups at 64k context.

In [ ]:
class NativeSparseAttention(nn.Module):
    """
    Simplified Native Sparse Attention (NSA).

    Three parallel attention paths combined by learned gates:
    1. Compressed — block-mean pool K/V, attend to summaries
    2. Selected  — score blocks, pick top-k, full attention on selected
    3. Sliding   — local window attention

    output = g_cmp * attn_compressed + g_sel * attn_selected + g_swa * attn_sliding
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        block_size: int = 8,
        top_k: int = 2,
        window_size: int = 16,
    ):
        """
        Args:
            d_model:     Input embedding dimension. Must be divisible by n_heads.
            n_heads:     Number of attention heads.
            block_size:  Block size for compression and selection.
            top_k:       Number of blocks to select per query.
            window_size: Sliding window size for the local path.
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.block_size = block_size
        self.top_k = top_k
        self.window_size = window_size

        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)

        # Learned gates for combining paths (per-head)
        self.gate = nn.Linear(d_model, n_heads * 3, bias=False)

    def _compressed_attention(
        self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor
    ) -> torch.Tensor:
        """
        Pool K/V into block summaries, then attend to the pooled representations.

        Args:
            Q: (B, n_heads, T, d_k)
            K: (B, n_heads, T, d_k)
            V: (B, n_heads, T, d_k)

        Returns:
            Context vectors, shape (B, n_heads, T, d_k).
        """
        B, H, T, D = K.shape
        b = self.block_size

        # Pad to multiple of block_size
        pad = (b - T % b) % b
        if pad > 0:
            K = F.pad(K, (0, 0, 0, pad))  # (B, H, T+pad, D)
            V = F.pad(V, (0, 0, 0, pad))

        T_padded = K.shape[2]
        n_blocks = T_padded // b

        # Block-mean pool K and V
        K_pooled = K.view(B, H, n_blocks, b, D).mean(dim=3)  # (B, H, n_blocks, D)
        V_pooled = V.view(B, H, n_blocks, b, D).mean(dim=3)  # (B, H, n_blocks, D)

        # Standard attention on pooled representations
        scores = torch.matmul(Q, K_pooled.transpose(-2, -1)) / math.sqrt(D)  # (B, H, T, n_blocks)
        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, V_pooled)  # (B, H, T, D)

    def _selected_attention(
        self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor
    ) -> torch.Tensor:
        """
        Score blocks by mean-pooled K, select top-k blocks, run full attention
        on the selected tokens only.

        Args:
            Q: (B, n_heads, T, d_k)
            K: (B, n_heads, T, d_k)
            V: (B, n_heads, T, d_k)

        Returns:
            Context vectors, shape (B, n_heads, T, d_k).
        """
        B, H, T, D = K.shape
        b = self.block_size

        pad = (b - T % b) % b
        if pad > 0:
            K = F.pad(K, (0, 0, 0, pad))
            V = F.pad(V, (0, 0, 0, pad))

        T_padded = K.shape[2]
        n_blocks = T_padded // b

        # Score each block via mean-pooled keys
        K_pooled = K.view(B, H, n_blocks, b, D).mean(dim=3)                   # (B, H, n_blocks, D)
        block_scores = torch.matmul(Q, K_pooled.transpose(-2, -1))            # (B, H, T, n_blocks)

        # Select top-k blocks per query position
        k = min(self.top_k, n_blocks)
        _, top_idx = block_scores.topk(k, dim=-1)  # (B, H, T, k)

        # Gather selected blocks' tokens
        # Expand block indices to token indices: each block -> b tokens
        token_idx = top_idx.unsqueeze(-1) * b + torch.arange(b, device=K.device)  # (B, H, T, k, b)
        token_idx = token_idx.view(B, H, T, k * b)                                # (B, H, T, k*b)
        token_idx = token_idx.clamp(max=T_padded - 1)

        # Gather K/V for selected tokens
        token_idx_exp = token_idx.unsqueeze(-1).expand(-1, -1, -1, -1, D)  # (B, H, T, k*b, D)
        K_exp = K.unsqueeze(2).expand(-1, -1, T, -1, -1)                   # (B, H, T, T_padded, D)
        V_exp = V.unsqueeze(2).expand(-1, -1, T, -1, -1)

        K_sel = torch.gather(K_exp, 3, token_idx_exp)  # (B, H, T, k*b, D)
        V_sel = torch.gather(V_exp, 3, token_idx_exp)

        # Attention on selected tokens
        scores = torch.matmul(Q.unsqueeze(3), K_sel.transpose(-2, -1)).squeeze(3) / math.sqrt(D)
        weights = F.softmax(scores, dim=-1)                       # (B, H, T, k*b)
        return torch.matmul(weights.unsqueeze(3), V_sel).squeeze(3)  # (B, H, T, D)

    def _sliding_attention(
        self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor
    ) -> torch.Tensor:
        """
        Standard causal sliding window attention.

        Args:
            Q, K, V: (B, n_heads, T, d_k)

        Returns:
            Context vectors, shape (B, n_heads, T, d_k).
        """
        T = Q.shape[2]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, H, T, T)

        # Causal + window mask
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=Q.device), diagonal=1)
        window = torch.tril(torch.ones(T, T, dtype=torch.bool, device=Q.device), diagonal=-(self.window_size))
        mask = causal | window
        scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, V)  # (B, H, T, D)

    def forward(
        self,
        x: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: Input tensor, shape (batch, seq_len, d_model).

        Returns:
            out:   Context vectors, shape (batch, seq_len, d_model).
            gates: Gate values per path, shape (batch, n_heads, seq_len, 3).
        """
        B, T, _ = x.shape

        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, T, d_k)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Three parallel attention paths
        a_cmp = self._compressed_attention(Q, K, V)  # (B, H, T, d_k)
        a_sel = self._selected_attention(Q, K, V)    # (B, H, T, d_k)
        a_swa = self._sliding_attention(Q, K, V)     # (B, H, T, d_k)

        # Learned gates: sigmoid for [0, 1] weighting
        g = self.gate(x).view(B, T, self.n_heads, 3).permute(0, 2, 1, 3)  # (B, H, T, 3)
        g = torch.sigmoid(g)
        g_cmp, g_sel, g_swa = g[..., 0:1], g[..., 1:2], g[..., 2:3]      # each (B, H, T, 1)

        # Gated combination
        context = g_cmp * a_cmp + g_sel * a_sel + g_swa * a_swa  # (B, H, T, d_k)

        context = context.transpose(1, 2).contiguous().view(B, T, -1)  # (B, T, d_model)
        out = self.W_o(context)
        return out, g

In [ ]:
torch.manual_seed(0)

batch, seq_len, d_model = 2, 64, 64
n_heads, block_size, top_k, window_size = 4, 8, 2, 16

x = torch.randn(batch, seq_len, d_model)
nsa = NativeSparseAttention(
    d_model=d_model, n_heads=n_heads,
    block_size=block_size, top_k=top_k, window_size=window_size,
)
out, gates = nsa(x)

print(f"Input  shape: {x.shape}")      # (2, 64, 64)
print(f"Output shape: {out.shape}")    # (2, 64, 64)
print(f"Gates  shape: {gates.shape}")  # (2, 4, 64, 3)

# Gate values show per-head, per-position weighting of the 3 paths
g_mean = gates.mean(dim=(0, 2))  # (n_heads, 3) — average gate per head
print(f"\nMean gate values per head (compressed, selected, sliding):")
for h in range(n_heads):
    g_cmp, g_sel, g_swa = g_mean[h].tolist()
    print(f"  Head {h}: cmp={g_cmp:.3f}, sel={g_sel:.3f}, swa={g_swa:.3f}")

# Cost comparison
n_blocks = seq_len // block_size
cost_full = seq_len * seq_len
cost_nsa = seq_len * (n_blocks + top_k * block_size + window_size)
print(f"\nAttention entries: full={cost_full}, NSA≈{cost_nsa} ({100 * cost_nsa / cost_full:.0f}%)")
print("\nShape checks passed.")

## 6. Gated Attention with QK-Norm

Two orthogonal improvements to the standard attention output:

1. **QK-Norm** — apply RMSNorm to Q and K before the dot product, preventing attention logit explosion at scale
2. **Output gate** — a learned sigmoid gate suppresses uninformative attention updates before the residual connection

$$\text{GatedAttn}(X) = \sigma(X W_g) \odot \text{Attention}(\text{RMSNorm}(Q),\, \text{RMSNorm}(K),\, V)$$

| Component | What it does | Cost |
|-----------|-------------|------|
| QK-Norm | Bounds attention logits regardless of `d_model` | Two norm ops — negligible |
| Gate $\sigma(X W_g)$ | Per-position, per-dimension suppression in $[0, 1]$ | $d_{\text{model}}^2$ params per layer |

> **Why QK-Norm?** At 8B+ scale, Dehghani et al. (2023) found attention logits exceeding 50,000 — near-one-hot weights that killed gradients. QK-Norm stabilized training across **three orders of magnitude** of learning rate variation.

In [ ]:
class GatedAttention(nn.Module):
    """
    Multi-head attention with QK-Norm and a learned output gate.

    QK-Norm applies RMSNorm to Q and K before the dot product, bounding
    attention logits regardless of scale. The sigmoid gate suppresses
    uninformative attention outputs before the residual connection.

    GatedAttn(X) = sigmoid(X @ W_g) * MHA(RMSNorm(Q), RMSNorm(K), V)
    """

    def __init__(self, d_model: int, n_heads: int, qk_norm: bool = True):
        """
        Args:
            d_model: Input embedding dimension. Must be divisible by n_heads.
            n_heads: Number of attention heads.
            qk_norm: Whether to apply RMSNorm to Q and K.
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.qk_norm = qk_norm

        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)

        # Learned output gate
        self.W_g = nn.Linear(d_model, d_model, bias=False)

        # Per-head RMSNorm for Q and K
        if qk_norm:
            self.q_norm = nn.RMSNorm(self.d_k)
            self.k_norm = nn.RMSNorm(self.d_k)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x:    Input tensor, shape (batch, seq_len, d_model).
            mask: Optional boolean mask, shape (seq_len, seq_len).

        Returns:
            out:     Gated context vectors, shape (batch, seq_len, d_model).
            weights: Attention weights, shape (batch, n_heads, seq_len, seq_len).
        """
        B, T, _ = x.shape

        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, n_heads, T, d_k)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # QK-Norm: normalize Q and K per head before dot product
        if self.qk_norm:
            Q = self.q_norm(Q)  # (B, n_heads, T, d_k) — norm on last dim
            K = self.k_norm(K)

        # Standard scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        context = torch.matmul(weights, V)

        context = context.transpose(1, 2).contiguous().view(B, T, -1)  # (B, T, d_model)
        attn_out = self.W_o(context)

        # Output gate: sigmoid in [0, 1], element-wise multiply
        gate = torch.sigmoid(self.W_g(x))  # (B, T, d_model)
        out = gate * attn_out               # (B, T, d_model)

        return out, weights

In [ ]:
torch.manual_seed(0)

batch, seq_len, d_model, n_heads = 2, 7, 64, 4

x = torch.randn(batch, seq_len, d_model)

gated = GatedAttention(d_model=d_model, n_heads=n_heads, qk_norm=True)
ungated_attn = GatedAttention(d_model=d_model, n_heads=n_heads, qk_norm=False)

out_gated, w_gated = gated(x)
out_ungated, w_ungated = ungated_attn(x)

print(f"Input  shape : {x.shape}")          # (2, 7, 64)
print(f"Output shape : {out_gated.shape}")  # (2, 7, 64)
print(f"Weights shape: {w_gated.shape}")    # (2, 4, 7, 7)

# Show gate suppression effect
gate_vals = torch.sigmoid(gated.W_g(x))  # (B, T, d_model)
print(f"\nGate stats: min={gate_vals.min():.3f}, max={gate_vals.max():.3f}, mean={gate_vals.mean():.3f}")
print(f"Output norm (gated)  : {out_gated.norm():.3f}")
print(f"Output norm (ungated): {out_ungated.norm():.3f}")

# QK-Norm effect: compare score magnitudes
Q = gated.W_q(x).view(batch, seq_len, n_heads, gated.d_k).transpose(1, 2)
K = gated.W_k(x).view(batch, seq_len, n_heads, gated.d_k).transpose(1, 2)
scores_raw = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(gated.d_k)
scores_normed = torch.matmul(gated.q_norm(Q), gated.k_norm(K).transpose(-2, -1)) / math.sqrt(gated.d_k)
print(f"\nScore magnitude — raw: max={scores_raw.abs().max():.1f}, normed: max={scores_normed.abs().max():.1f}")
print("QK-Norm bounds the scores, preventing softmax saturation.")

## 7. Hybrid Attention

Not every layer needs full $O(n^2)$ attention. Hybrid architectures interleave cheap **linear attention** layers ($O(n \cdot d^2)$) with expensive **full attention** layers, allocating quadratic compute only where global context is critical.

$$\text{Linear:}\; O = Q (K^\top V) \qquad\text{vs.}\qquad \text{Standard:}\; O = (Q K^\top) V$$

| Layer type | Cost | What it captures |
|-----------|------|-----------------|
| Linear attention | $O(n \cdot d^2)$ | Local patterns, syntax, short-range dependencies |
| Full (gated) attention | $O(n^2 \cdot d)$ | Global context, long-range coreference |

> **The associativity trick:** $(QK^\top)V = Q(K^\top V)$. Compute the $d_k \times d_v$ inner product first, multiply by each query — never materialize the $n \times n$ matrix. Winning ratios cluster around **6–7 linear layers per 1 full attention layer**.

In [ ]:
class LinearAttention(nn.Module):
    """
    Simplified linear attention using the associativity trick.

    Instead of materializing the n×n attention matrix (Q @ K.T) @ V,
    computes Q @ (K.T @ V) — cost O(n * d^2) instead of O(n^2 * d).
    No softmax normalization — works for local patterns but not precise
    long-range key-value lookup.
    """

    def __init__(self, d_model: int, n_heads: int):
        """
        Args:
            d_model: Input embedding dimension. Must be divisible by n_heads.
            n_heads: Number of attention heads.
        """
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_k, d_model, bias=False)
        self.norm = nn.RMSNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor, shape (batch, seq_len, d_model).

        Returns:
            Output tensor, shape (batch, seq_len, d_model).
        """
        B, T, _ = x.shape

        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, T, d_k)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Feature map: ELU + 1 ensures non-negative "attention weights"
        Q = F.elu(Q) + 1  # (B, H, T, d_k)
        K = F.elu(K) + 1

        # Associativity trick: Q @ (K.T @ V) instead of (Q @ K.T) @ V
        KV = torch.matmul(K.transpose(-2, -1), V)  # (B, H, d_k, d_k) — the small matrix
        context = torch.matmul(Q, KV)                # (B, H, T, d_k) — no n×n matrix!

        # Normalize by sum of keys (replaces softmax normalization)
        Z = Q.sum(dim=-1, keepdim=True) * K.sum(dim=-2, keepdim=True).sum(dim=-1, keepdim=True)
        context = context / (Z + 1e-6)

        context = context.transpose(1, 2).contiguous().view(B, T, -1)
        return self.norm(self.W_o(context))  # RMSNorm for stability


class HybridAttentionBlock(nn.Module):
    """
    Hybrid attention block interleaving linear and full attention layers.

    Assigns each layer to linear or full attention based on the ratio:
    every (ratio+1)-th layer gets full GatedAttention, the rest get
    LinearAttention. This allocates O(n^2) compute only where needed.
    """

    def __init__(self, d_model: int, n_heads: int, n_layers: int, ratio: int = 3):
        """
        Args:
            d_model:  Input embedding dimension.
            n_heads:  Number of attention heads.
            n_layers: Total number of layers.
            ratio:    Number of linear layers per full attention layer.
                      E.g., ratio=3 → [linear, linear, linear, full, linear, ...].
        """
        super().__init__()
        self.n_layers = n_layers
        self.ratio = ratio

        layers = []
        self.layer_types = []
        for i in range(n_layers):
            if (i + 1) % (ratio + 1) == 0:
                layers.append(GatedAttention(d_model, n_heads, qk_norm=True))
                self.layer_types.append("full")
            else:
                layers.append(LinearAttention(d_model, n_heads))
                self.layer_types.append("linear")
        self.layers = nn.ModuleList(layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor, shape (batch, seq_len, d_model).

        Returns:
            Output tensor, shape (batch, seq_len, d_model).
        """
        for layer, ltype in zip(self.layers, self.layer_types):
            if ltype == "full":
                out, _ = layer(x)  # GatedAttention returns (out, weights)
            else:
                out = layer(x)     # LinearAttention returns out only
            x = x + out  # residual connection
        return x

In [ ]:
torch.manual_seed(0)

batch, seq_len, d_model = 2, 16, 64
n_heads, n_layers, ratio = 4, 8, 3  # 6 linear + 2 full

x = torch.randn(batch, seq_len, d_model)
hybrid = HybridAttentionBlock(d_model=d_model, n_heads=n_heads, n_layers=n_layers, ratio=ratio)
out = hybrid(x)

print(f"Input  shape: {x.shape}")   # (2, 16, 64)
print(f"Output shape: {out.shape}") # (2, 16, 64)

# Show layer assignment
n_linear = sum(1 for t in hybrid.layer_types if t == "linear")
n_full = sum(1 for t in hybrid.layer_types if t == "full")
print(f"\nLayer layout ({n_layers} total: {n_linear} linear + {n_full} full):")
for i, ltype in enumerate(hybrid.layer_types):
    marker = "■ FULL (GatedAttention)" if ltype == "full" else "□ linear"
    print(f"  Layer {i}: {marker}")

# Parameter comparison: linear vs full layers
params_linear = sum(p.numel() for p in hybrid.layers[0].parameters())
params_full = sum(p.numel() for p in hybrid.layers[ratio].parameters())
params_total = sum(p.numel() for p in hybrid.parameters())
print(f"\nParams per layer: linear={params_linear:,}, full={params_full:,}")
print(f"Total params: {params_total:,}")
print("\nShape checks passed.")

## 8. KV Cache Comparison

The dominant inference bottleneck in LLMs is KV cache memory. Each variant addresses this differently. Below we compute cache sizes for a realistic config: 32 layers, `d_model=4096`, `d_k=128`, float16 — the approximate shape of LLaMA 3 8B.

In [ ]:
# KV cache comparison across variants (LLaMA 3 8B-scale config)
n_layers = 32
n_heads = 32
d_k = 128
d_latent = 512
n_kv_heads_gqa = 8
w = 4096
bytes_per_elem = 2  # float16

seq_lengths = [1024, 8192, 32768, 131072]

print("KV Cache Memory (GB, float16)")
print(f"Config: {n_layers} layers, {n_heads} heads, d_k={d_k}")
print("=" * 75)
header = f"{'T':>8}  {'MHA':>8}  {'GQA-8':>8}  {'MLA':>8}  {'SWA':>8}"
print(header)
print("-" * 75)

for T in seq_lengths:
    # MHA: 2 * n_layers * n_heads * T * d_k
    mha_gb = 2 * n_layers * n_heads * T * d_k * bytes_per_elem / 1e9
    # GQA: 2 * n_layers * n_kv_heads * T * d_k
    gqa_gb = 2 * n_layers * n_kv_heads_gqa * T * d_k * bytes_per_elem / 1e9
    # MLA: 1 * n_layers * T * d_latent (just the latent c, no per-head)
    mla_gb = n_layers * T * d_latent * bytes_per_elem / 1e9
    # SWA: 2 * n_layers * n_heads * min(T, w) * d_k (bounded by window)
    swa_gb = 2 * n_layers * n_heads * min(T, w) * d_k * bytes_per_elem / 1e9

    print(f"{T:>8,}  {mha_gb:>7.1f}G  {gqa_gb:>7.1f}G  {mla_gb:>7.1f}G  {swa_gb:>7.1f}G")

print("-" * 75)
print("SWA cache is bounded at w=4096 regardless of sequence length.")
print("MLA caches only the latent (d_latent=512), not per-head K/V.")

In [ ]:
# Visualize KV cache scaling
fig, ax = plt.subplots(figsize=(10, 5))

T_range = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]

for label, calc_fn, color, ls in [
    ("MHA (32 heads)", lambda T: 2 * 32 * 32 * T * 128 * 2 / 1e9, "#1E3A8A", "-"),
    ("GQA (8 KV heads)", lambda T: 2 * 32 * 8 * T * 128 * 2 / 1e9, "#3B82F6", "-"),
    ("MLA (d_latent=512)", lambda T: 32 * T * 512 * 2 / 1e9, "#10B981", "-"),
    ("SWA (w=4096)", lambda T: 2 * 32 * 32 * min(T, 4096) * 128 * 2 / 1e9, "#F59E0B", "--"),
]:
    values = [calc_fn(T) for T in T_range]
    ax.plot(T_range, values, label=label, color=color, linewidth=2, linestyle=ls)

ax.set_xlabel("Sequence Length (T)", fontsize=12)
ax.set_ylabel("KV Cache Memory (GB, float16)", fontsize=12)
ax.set_title("KV Cache Scaling — 32 layers, d_k=128", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xscale("log", base=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()